## Statistical method

### Data preprocessing

In [7]:
import pandas as pd
import numpy as np

# 1. Load your raw dataset
# Replace 'raw_bnpl_data.csv' with your actual filename
df = pd.read_csv('../data/raw/raw_bnpl_data.csv')

# 2. Date Feature Engineering
if 'transaction_date' in df.columns:
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
    
    df['transaction_year'] = df['transaction_date'].dt.year
    df['transaction_month'] = df['transaction_date'].dt.month
    df['transaction_day'] = df['transaction_date'].dt.day
    df['transaction_dayofweek'] = df['transaction_date'].dt.dayofweek  # 0=Monday
    df['transaction_is_weekend'] = df['transaction_date'].dt.dayofweek.isin([5, 6]).astype(int)
    print("✅ Date features extracted.")

# 3. Handle Skewed Distributions (Log Transformations)
# These columns typically have long tails in credit data
log_cols = ['debt_to_income_ratio', 'purchase_amount', 'monthly_income']
log1p_cols = ['missed_payments', 'repayment_delay_days']

# Apply standard Log (for values > 0)
for col in log_cols:
    if col in df.columns:
        # Using log1p here is safer in case of zeros
        df[f"{col}_log"] = np.log1p(df[col])

# Apply Log1p (specifically for counts/days that might be 0)
for col in log1p_cols:
    if col in df.columns:
        df[f"{col}_log1p"] = np.log1p(df[col])

print("✅ Log transformations applied.")

# 4. Save the preprocessed file
df.to_csv('../data/interim/01_preprocessed.csv', index=False)
print("✅ Saved as 01_preprocessed.csv")

# Quick check
print(df.head())

✅ Date features extracted.
✅ Log transformations applied.
✅ Saved as 01_preprocessed.csv
   user_id  age employment_type  monthly_income  credit_score  \
0        1   56        Salaried        68529.50           552   
1        2   19         Student         7247.85           300   
2        3   20   Self-Employed        41582.26           471   
3        4   21        Salaried        14423.46           300   
4        5   43        Salaried        42845.50           512   

   purchase_amount product_category  bnpl_installments  repayment_delay_days  \
0          5000.00      Electronics                 12                    13   
1          1073.23          Fashion                 12                    13   
2          5000.00      Electronics                  3                    19   
3          4076.83           Sports                  6                    18   
4          5000.00      Electronics                  9                     0   

   missed_payments  ...  transaction_ye

### Data Loading & Variable Setup
In this step, we will load your preprocessed dataset and organize the features into the categories we identified earlier (Categorical, Ordinal, and Ratio).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load the preprocessed data 
# Replace '01_preprocessed.csv' with the actual path to your file
df = pd.read_csv('../data/interim/01_preprocessed.csv') 

# 2. Define variable groups based on your notebook classification
categorical_cols = ['employment_type', 'product_category', 'location']
ordinal_cols = ['customer_segment']
# Using Ratio/Interval variables for numerical tests
numerical_cols = [
    'age', 'monthly_income', 'purchase_amount', 'bnpl_installments', 
    'repayment_delay_days', 'missed_payments', 'app_usage_frequency', 
    'debt_to_income_ratio', 'risk_score', 'credit_score'
]
target = 'default_flag'

# 3. Quick Data Quality Check
print("Dataset Shape:", df.shape)
print("\nTarget Distribution (default_flag):")
print(df[target].value_counts(normalize=True))

# 4. Display the first few rows of the grouped columns to verify
print("\nSample Data (Categorical & Target):")
print(df[categorical_cols + [target]].head())

Dataset Shape: (10345, 27)

Target Distribution (default_flag):
default_flag
0    0.609473
1    0.390527
Name: proportion, dtype: float64

Sample Data (Categorical & Target):
  employment_type product_category   location  default_flag
0        Salaried      Electronics  Australia             0
1         Student          Fashion        USA             0
2   Self-Employed      Electronics  Australia             0
3        Salaried           Sports    Germany             1
4        Salaried      Electronics      India             0


### Analysis of Results
Here is what your output tells us about the dataset:

1. Dataset Volume (10,345 records, 27 columns):

- You have a substantial amount of data, which is excellent for statistical testing.
- The increase from the original 17 features to 27 columns suggests that your preprocessing (Step 01) successfully created new features (like year, month, day) or perhaps applied some encoding. This gives us more "dimensions" to test for credit risk.

2. Target Distribution (39.05% Default Rate):

- High Statistical Power: In typical bank data, the "Default" rate is often very low (1% to 5%), which makes it hard to find patterns. A 39% default rate is actually very "healthy" for analysis because it provides enough examples of both classes (0 and 1) to make our $p$-values very reliable.
- Balance: Your data is relatively balanced (61% vs 39%). You won't need to worry as much about extreme bias in your statistical tests.

3. Data Structure:

- The sample data shows that your categories like employment_type (Salaried, Student, etc.) and location (Australia, USA, etc.) are clean and ready for grouping.

Conclusion: The data is properly loaded and the target variable is well-represented. We can now proceed to see if these categories actually "matter" when it comes to predicting a default.